# Rules from the data

Goal: two sets of rules, each chosen by looking at the data first, and the labels they produce.

1. **H / A rule** — on frames where the ball is visible: which team *clearly* has it.
2. **Dead-ball rules** — on every frame: is play stopped.

No model and no ground truth. Every threshold is chosen from a plot or table shown just above it,
and each part ends with a list of frames to check by eye in the tactical video
(frame *i* of the video = row *i*; time = *i* / 25 s).

## 0. Setup

In [ ]:
import io
import os
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = os.path.abspath('.')
sys.path.insert(0, ROOT)
sys.path.insert(0, os.path.join(ROOT, 'starter'))

import games
import load_smart_tagging
from baseline import team_of_each_track
from utils import draw_pitch

warnings.filterwarnings('ignore', category=RuntimeWarning)

GAME = games.GAME
FPS = games.GAMES[GAME]['fps']
N = games.GAMES[GAME]['n_video_frames']
L, W = games.GAMES[GAME]['pitch']
HALVES = games.half_video_blocks(GAME)
KICKOFF_2ND = 113742  # FAQ.md: the 2nd half is graded from 110,808 but kicks off here
H, A = 0, 1
TEAM_NAME = {H: 'Barça', A: 'Athletic'}

graded = np.zeros(N, bool)
for _, first, last in HALVES:
    graded[first:last + 1] = True


def runs_of(mask):
    # [(start, end_exclusive)] of the True runs of a boolean array.
    edges = np.diff(np.r_[0, mask.astype(np.int8), 0])
    return list(zip(np.flatnonzero(edges == 1), np.flatnonzero(edges == -1)))


def video_time(frame):
    s = frame / FPS
    return f'{int(s // 3600)}:{int(s // 60) % 60:02d}:{s % 60:05.2f}'


def pct(x, qs=(10, 50, 90)):
    return ', '.join(f'p{q} {np.nanpercentile(x, q):.1f}' for q in qs)

In [ ]:
CACHE = os.path.join(games.DATA_DIR, 'cache', 'atd_explore.npz')  # written by explore_tracking.ipynb


def load_tracking():
    if os.path.isfile(CACHE):
        with np.load(CACHE) as z:
            return {k: z[k] for k in z.files}
    meta_path, raw_path = games.require(['atd_meta', 'atd_raw'])
    with open(raw_path) as handle:
        text = handle.read().replace(':', ',').replace(';', ',')
    table = pd.read_csv(io.StringIO(text), header=None, dtype=np.float64).to_numpy()
    n_slots = (table.shape[1] - 4) // 3
    scale = np.array([L, W])
    players = table[:, 1:1 + 3 * n_slots].reshape(N, n_slots, 3)
    data = {'ball_xy': table[:, -3:-1] * scale, 'ball_s_file': table[:, -1],
            'player_xy': (players[:, :, :2] * scale).astype(np.float32),
            'player_s': players[:, :, 2].astype(np.float32), 'team': team_of_each_track(meta_path)}
    os.makedirs(os.path.dirname(CACHE), exist_ok=True)
    np.savez(CACHE, **data)
    return data


trk = load_tracking()
ball_xy, player_xy, player_s, team = trk['ball_xy'], trk['player_xy'], trk['player_s'], trk['team']
active = ~np.isnan(player_xy[:, :, 0])
visible = ~np.isnan(ball_xy[:, 0])

# Ball speed from positions (the file's ball speed field is unusable).
ball_v = np.full(N, np.nan)
ball_v[2:-2] = np.linalg.norm(ball_xy[4:] - ball_xy[:-4], axis=1) * FPS / 4
# Signed distance to the nearest line: positive inside the pitch, negative outside.
d_line = np.minimum.reduce([ball_xy[:, 0], L - ball_xy[:, 0], ball_xy[:, 1], W - ball_xy[:, 1]])
# Mean speed of all tracked players, averaged over 1 s.
mean_speed = pd.Series(np.nanmean(np.where(active, player_s, np.nan), axis=1)) \
    .rolling(FPS, center=True, min_periods=1).mean().to_numpy()

events = load_smart_tagging.load()
print(f'ball visible on {visible[graded].mean():.1%} of graded frames')

## 1. Attacking direction

Fixed per team per half. Found from the goalkeeper slots (24 Barça, 46 Athletic) and checked
against each team's average position over the half. Not used by the rules below, but needed by
any player-movement feature later.

In [ ]:
GK_SLOT = {H: 24 - 1, A: 46 - 1}
attack_dir = {}
for half, first, last in HALVES:
    span = slice(max(first, KICKOFF_2ND) if half == 2 else first, last + 1)
    for t in (H, A):
        gk_x = np.nanmedian(player_xy[span, GK_SLOT[t], 0])
        team_x = np.nanmean(player_xy[span][:, team == t, 0])
        d_gk = 1 if gk_x < L / 2 else -1     # keeper near x=0 -> team attacks towards +x
        d_team = 1 if team_x < L / 2 else -1
        attack_dir[(t, half)] = d_gk
        print(f'half {half} {TEAM_NAME[t]:<8}: keeper median x {gk_x:5.1f}, team mean x {team_x:5.1f} '
              f'-> attacks {"+x" if d_gk > 0 else "-x"}  (team-mean check {"agrees" if d_gk == d_team else "DISAGREES"})')

## 2. H / A rule

A frame gets **H** (or **A**, mirrored) when the ball is visible, inside the pitch, not flying,
a Barça player is close to it and clearly closer than any Athletic player — for a few frames in a
row. Everything else stays unlabelled.

Four numbers to choose: **how close** (`R`), **how much closer than the opponent** (`G`),
**how slow** (`V`), **for how long** (`K`). First the data.

In [ ]:
def nearest_by_team(xy, chunk=20000):
    dH = np.full(N, np.inf)
    dA = np.full(N, np.inf)
    for s in range(0, N, chunk):
        d = np.linalg.norm(player_xy[s:s + chunk] - xy[s:s + chunk, None, :], axis=2)
        d = np.where(np.isnan(d), np.inf, d)
        dH[s:s + chunk] = d[:, team == H].min(axis=1)
        dA[s:s + chunk] = d[:, team == A].min(axis=1)
    return dH, dA


dH, dA = nearest_by_team(ball_xy)
d_near = np.minimum(dH, dA)
d_gap = np.abs(dA - dH)  # how much further the nearest opponent is
usable = visible & graded & (d_line > 0) & ~np.isnan(ball_v) & np.isfinite(d_near)
print(f'usable frames (ball visible, inside, speed known): {usable.sum()}')

### 2a. How close is the nearest player, and how fast is the ball?

Left: distance from the ball to the nearest player (any team). Right: the same against ball
speed. A player with the ball at their feet should show up as a cluster of **small distance and
low speed**; passes as **high speed, larger distance**.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4))
axes[0].hist(np.clip(d_near[usable], 0, 10), bins=100, color='tab:green')
axes[0].set_xlabel('nearest player to ball (m), clipped at 10')
axes[0].set_title('nearest-player distance')

axes[1].hist(np.clip(ball_v[usable], 0, 35), bins=140, color='tab:blue')
axes[1].set_xlabel('ball speed (m/s), clipped at 35')
axes[1].set_title('ball speed')

h = axes[2].hist2d(np.clip(d_near[usable], 0, 8), np.clip(ball_v[usable], 0, 30),
                   bins=[80, 60], cmap='magma', norm=plt.matplotlib.colors.LogNorm())
axes[2].set_xlabel('nearest player (m)')
axes[2].set_ylabel('ball speed (m/s)')
axes[2].set_title('distance vs speed (log colour)')
plt.colorbar(h[3], ax=axes[2])
plt.tight_layout()
plt.show()

print(f'nearest-player distance (m): {pct(d_near[usable], (10, 25, 50, 75, 90))}')
for v_max in [2, 4, 6, 8]:
    slow = usable & (ball_v < v_max)
    print(f'  when ball slower than {v_max} m/s ({slow.mean() / usable.mean():.0%} of usable): '
          f'{pct(d_near[slow], (25, 50, 75, 90))}')

### 2b. Is the nearest player clearly closer than the nearest opponent?

When someone is close to a slow ball, how much further away is the nearest player of the other
team?

In [ ]:
close_slow = usable & (d_near < 2) & (ball_v < 6)
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.hist(np.clip(d_gap[close_slow], 0, 15), bins=75, color='tab:purple')
ax.set_xlabel('nearest opponent minus nearest player (m), clipped at 15')
ax.set_title('separation when a player is within 2 m of a slow ball')
plt.show()
print(f'separation (m): {pct(d_gap[close_slow], (10, 25, 50, 75, 90))}')
for g in [0.5, 1, 1.5, 2, 3]:
    print(f'  separation > {g} m: {np.mean(d_gap[close_slow] > g):.0%}')

### 2c. Choosing the four numbers

Read off the plots above (see the conclusions under the sensitivity table for the reasoning):

In [ ]:
RULE_HA = dict(
    R=1.5,   # m: nearest player within this distance
    G=1.5,   # m: nearest opponent at least this much further away
    V=6.0,   # m/s: ball slower than this (not a pass or shot in flight)
    K=5,     # frames (0.2 s): condition holds for at least this many frames in a row
)


def label_ha(R, G, V, K):
    ok = usable & (d_near < R) & (d_gap > G) & (ball_v < V)
    side = np.where(dH < dA, H, A)
    lab = np.full(N, -1, dtype=np.int8)
    for t in (H, A):
        for a, b in runs_of(ok & (side == t)):
            if b - a >= K:
                lab[a:b] = t
    return lab


ha = label_ha(**RULE_HA)
n_lab = np.sum(ha >= 0)
print(f'labelled frames: {n_lab} ({n_lab / graded.sum():.1%} of graded, {n_lab / usable.sum():.1%} of usable)')
print(f'  H {np.sum(ha == H)}  A {np.sum(ha == A)}  (H share {np.mean(ha[ha >= 0] == H):.0%})')
stretches = [(a, b, ha[a]) for t in (H, A) for a, b in runs_of(ha == t)]
lens = np.array([b - a for a, b, _ in stretches]) / FPS
print(f'labelled stretches: {len(stretches)}, length (s): {pct(lens)}')

### 2d. How much do the numbers matter?

In [ ]:
rows = []
for R in [1.0, 1.5, 2.0]:
    for G in [1.0, 1.5, 2.5]:
        for K in [3, 5, 10]:
            lab = label_ha(R=R, G=G, V=RULE_HA['V'], K=K)
            n = np.sum(lab >= 0)
            rows.append({'R': R, 'G': G, 'K': K, 'labelled': n,
                         '% of usable': round(100 * n / usable.sum(), 1),
                         'H share %': round(100 * np.mean(lab[lab >= 0] == H), 1),
                         'stretches': len(runs_of(lab == H)) + len(runs_of(lab == A))})
sens = pd.DataFrame(rows)
sens.pivot_table(index=['R', 'G'], columns='K', values='% of usable')

In [ ]:
# Does the choice change WHICH team gets labelled, or only how many frames?
base = label_ha(**RULE_HA)
for R, G in [(1.0, 1.0), (2.0, 2.5)]:
    other = label_ha(R=R, G=G, V=RULE_HA['V'], K=RULE_HA['K'])
    both = (base >= 0) & (other >= 0)
    print(f'R={R}, G={G}: shares {both.sum()} labelled frames with the chosen rule; '
          f'same team on {np.mean(base[both] == other[both]):.1%} of them')

### 2e. Frames to check in the video

Twelve random labelled stretches. Open the tactical video at the time shown: the named team should
clearly have the ball at a player's feet.

In [ ]:
rng = np.random.default_rng(0)
pick = rng.choice(len(stretches), 12, replace=False)
pd.DataFrame([{'label': TEAM_NAME[stretches[i][2]], 'start_frame': stretches[i][0],
               'video_time': video_time(stretches[i][0]),
               'length_s': (stretches[i][1] - stretches[i][0]) / FPS} for i in sorted(pick)])

## 3. Dead-ball rules

Four rules, each from a signal the exploration showed. Each one marks a **start** of a stoppage;
the stoppage ends at the **restart**.

| Rule | Starts a stoppage when… | Why (from the exploration) |
|---|---|---|
| R1 ball out | the ball is seen > 0.5 m outside the pitch | 67 episodes; certain when seen |
| R2 lost at a line | the ball disappears within 2 m of a line (or outside) | gaps start near lines 3× more often than chance |
| R3 stationary | the visible ball is still (< 0.5 m/s) for ≥ 1 s | 42 episodes, all on set-piece spots |
| R4 set-piece tag | a set-piece tag starts | players slow and ball still/missing in the ~10 s before |

**Restart** (end of the stoppage): the first time after the start that the ball is visible,
inside the pitch and moving (> 3 m/s) for at least 0.4 s. If none is found within 120 s, the
stoppage ends when the ball is next seen inside the pitch.

In [ ]:
RULE_D = dict(
    out_margin=0.5,      # m past a line = out (R1)
    lost_near_line=2.0,  # m from a line when the ball disappears (R2)
    still_speed=0.5,     # m/s (R3)
    still_min_s=1.0,     # s (R3)
    tag_before_s=8.0,    # s of dead ball assumed before a set-piece tag start (R4)
    restart_speed=3.0,   # m/s: ball moving again
    restart_hold=10,     # frames the restart condition must hold
    max_stop_s=120.0,
)


def find_restart(start, p=RULE_D):
    # First frame >= start where the ball is visible, inside and moving for restart_hold frames.
    end = min(start + int(p['max_stop_s'] * FPS), N)
    moving = visible[start:end] & (d_line[start:end] > 0) & (ball_v[start:end] > p['restart_speed'])
    for a, b in runs_of(moving):
        if b - a >= p['restart_hold']:
            return start + a
    inside = np.flatnonzero(visible[start:end] & (d_line[start:end] > 0))
    return start + inside[0] if len(inside) else end


def dead_candidates(p=RULE_D):
    starts = []  # (frame, rule, frame the stoppage is certainly still on)
    out = visible & graded & (d_line < -p['out_margin'])
    for a, b in runs_of(out):
        starts.append((a, 'R1 ball out', b))
    for a, b in runs_of(~visible & graded):
        last = a - 1
        if last >= 0 and visible[last] and d_line[last] < p['lost_near_line']:
            starts.append((last, 'R2 lost at a line', b))
    still = visible & graded & (ball_v < p['still_speed'])
    for a, b in runs_of(still):
        if b - a >= p['still_min_s'] * FPS:
            starts.append((a, 'R3 stationary', b))
    for ev in events:
        if (ev['code'] or '').strip().upper() == 'SET PIECES' and graded[min(ev['start_frame'], N - 1)]:
            s = ev['start_frame']
            starts.append((max(s - int(p['tag_before_s'] * FPS), 0), 'R4 set-piece tag', s))
    return sorted(starts)


def build_stoppages(p=RULE_D):
    spans = []
    for start, rule, still_on in dead_candidates(p):
        end = find_restart(still_on, p)
        spans.append([start, max(end, still_on), {rule}])
    # The 2nd half before kickoff is dead by definition.
    spans.append([HALVES[1][1], KICKOFF_2ND, {'kickoff'}])
    spans.sort(key=lambda s: s[0])
    merged = []
    for s in spans:
        if merged and s[0] <= merged[-1][1] + 2 * FPS:
            merged[-1][1] = max(merged[-1][1], s[1])
            merged[-1][2] |= s[2]
        else:
            merged.append(s)
    dead = np.zeros(N, bool)
    for a, b, _ in merged:
        dead[a:b] = True
    return merged, dead & graded


stoppages, dead = build_stoppages()
dur = np.array([(b - a) / FPS for a, b, _ in stoppages])
minutes = graded.sum() / FPS / 60
print(f'{len(stoppages)} stoppages, dead on {dead[graded].mean():.1%} of graded frames '
      f'({dead.sum() / FPS / 60:.1f} of {minutes:.1f} min)')
print(f'stoppage length (s): {pct(dur, (10, 50, 90))}, max {dur.max():.0f}')

### 3a. Which rules fire, and how often do they agree?

In [ ]:
combo = pd.Series([' + '.join(sorted(r for r in rules)) for _, _, rules in stoppages]).value_counts()
per_rule = {r: sum(r in rules for _, _, rules in stoppages)
            for r in ['R1 ball out', 'R2 lost at a line', 'R3 stationary', 'R4 set-piece tag', 'kickoff']}
print('stoppages containing each rule:', per_rule)
print()
print('combinations:')
print(combo.to_string())

### 3b. How much does each rule add?

Dead share and stoppage count with one rule switched off at a time. A rule that adds a lot on its
own, with little agreement from the others, is the one to check most carefully in the video.

In [ ]:
def without(rule_name):
    p = dict(RULE_D)
    keep = dead_candidates
    rows = [c for c in keep(p) if c[1] != rule_name]
    spans = [[s, max(find_restart(o, p), o), {r}] for s, r, o in rows]
    spans.append([HALVES[1][1], KICKOFF_2ND, {'kickoff'}])
    spans.sort(key=lambda s: s[0])
    merged = []
    for s in spans:
        if merged and s[0] <= merged[-1][1] + 2 * FPS:
            merged[-1][1] = max(merged[-1][1], s[1])
        else:
            merged.append(s)
    d = np.zeros(N, bool)
    for a, b, _ in merged:
        d[a:b] = True
    return len(merged), (d & graded)[graded].mean()


rows = [{'rules': 'all', 'stoppages': len(stoppages), 'dead share': f'{dead[graded].mean():.1%}'}]
for r in ['R1 ball out', 'R2 lost at a line', 'R3 stationary', 'R4 set-piece tag']:
    n, share = without(r)
    rows.append({'rules': f'without {r}', 'stoppages': n, 'dead share': f'{share:.1%}'})
pd.DataFrame(rows)

### 3c. R2 is the uncertain one: what happens after the ball is lost at a line?

If those gaps are real stoppages, the ball should come back **at a set-piece spot or on the line**
and **start from still**. If they are live play, it should come back somewhere ordinary, moving.

In [ ]:
rows = []
for a, b in runs_of(~visible & graded):
    last = a - 1
    if last < 0 or not visible[last] or b >= N or not visible[b]:
        continue
    rows.append({
        'lost_at_line': d_line[last] < RULE_D['lost_near_line'],
        'gap_s': (b - a) / FPS,
        'back_at_line_m': d_line[b],
        'back_speed': np.nanmedian(ball_v[b:b + 5]),
        'players_slow_in_gap': np.nanmean(mean_speed[a:b]) < 1.3,
        'tag_in_gap': any(a - 2 * FPS <= ev['start_frame'] <= b + 5 * FPS for ev in events
                          if (ev['code'] or '').strip().upper() == 'SET PIECES'),
    })
gaps = pd.DataFrame(rows)
summary = gaps.groupby('lost_at_line').agg(
    gaps=('gap_s', 'size'),
    gap_s_median=('gap_s', 'median'),
    back_within_2m_of_line=('back_at_line_m', lambda s: np.mean(s < 2)),
    back_speed_median=('back_speed', 'median'),
    players_slow=('players_slow_in_gap', 'mean'),
    tag_nearby=('tag_in_gap', 'mean'),
).round(2)
summary.index = summary.index.map({True: 'lost within 2 m of a line', False: 'lost elsewhere'})
summary

### 3d. Stoppages to check in the video

A random sample from each rule combination. Open the video a few seconds before `video_time`: play
should stop around there and restart around `restart_time`.

In [ ]:
rng = np.random.default_rng(1)
sample = []
by_combo = {}
for i, (a, b, rules) in enumerate(stoppages):
    by_combo.setdefault(' + '.join(sorted(rules)), []).append(i)
for combo_name, idx in by_combo.items():
    for i in rng.choice(idx, min(3, len(idx)), replace=False):
        a, b, rules = stoppages[i]
        sample.append({'rules': combo_name, 'start_frame': a, 'video_time': video_time(a),
                       'restart_time': video_time(b), 'length_s': (b - a) / FPS})
pd.DataFrame(sample).sort_values('start_frame').reset_index(drop=True)

### 3e. Ten minutes with both rule sets on one axis

In [ ]:
def overview(first, minutes=10):
    last = first + minutes * 60 * FPS
    t = np.arange(first, last)
    x = (t - first) / FPS / 60
    fig, ax = plt.subplots(figsize=(16, 3.5))
    ax.plot(x, mean_speed[t], color='black', lw=0.7, label='mean player speed (1 s)')
    ax.fill_between(x, 0, 0.35, where=visible[t], color='tab:green', alpha=0.4, label='ball visible')
    ax.fill_between(x, 0.4, 0.75, where=ha[t] == H, color='tab:blue', label='rule: H')
    ax.fill_between(x, 0.4, 0.75, where=ha[t] == A, color='tab:red', label='rule: A')
    ax.fill_between(x, 0.8, 7, where=dead[t], color='grey', alpha=0.25, label='rule: dead')
    ax.set_ylim(0, 7)
    ax.set_xlabel(f'minutes from frame {first} (video {video_time(first)})')
    ax.set_ylabel('m/s')
    ax.legend(loc='upper right', fontsize=8, ncol=5)
    plt.tight_layout()
    plt.show()


overview(KICKOFF_2ND)
overview(140000)

## 4. Summary

### H / A rule
- **What the data shows.** The nearest player to the ball is most often ~0.8 m away; when the ball
  is slow, half the time someone is within 1.2 m. The distance-vs-speed plot has one dense cluster:
  **distance under ~1.5 m, ball slower than ~6 m/s** — the ball at a player's feet. Ball speed has
  no clean split between dribbling and passing, and the separation from the nearest opponent has
  no natural cut either (flat histogram).
- **Chosen:** R = 1.5 m and V = 6 m/s (the edges of that cluster), G = 1.5 m (keeps 84% of
  close-and-slow frames), K = 5 frames (0.2 s).
- **Result:** 25,394 labelled frames — 28% of frames with a usable ball, 16% of graded frames.
  H 62%, A 38%. Labelled stretches are short (median 0.8 s): the rule catches the moments a
  player clearly has the ball, not whole possessions — which is what it is for.
- **Are the numbers arbitrary?** They change *how many* frames get a label (18–35% of usable
  across the grid), but **never which team**: across the settings tested, the labelled team agrees
  on 100% of shared frames. The choice affects quantity, not correctness.
- **To check:** the 12 stretches in 2e, in the video.

### Dead-ball rules
- **Result:** 101 stoppages, dead on **28.4%** of graded frames (30 of 106 min), median length
  14 s. A realistic share is roughly 35–40%, so this is in the right range but still low.
- **What each rule adds** (dead share without it): set-piece tags matter most (28.4% → 21.6%),
  then lost-at-a-line (→ 25.0%), stationary (→ 25.8%), ball out (→ 27.2%; mostly overlaps the
  others).
- **Single-rule stoppages are the ones to check first:** 18 from *stationary* alone, 14 from a
  *tag* alone, 6 from *lost at a line* alone. A stationary ball can be live (a goalkeeper holding
  it is live play).
- **Lost at a line is about half right:** when the ball disappears within 2 m of a line it comes
  back near a line 48% of the time (18% otherwise), with a set-piece tag nearby 52% (22%) and
  players slow 57% (44%). Clearly more stoppage-like than other gaps, but far from certain.
- **Visible misses:** in the overview plots there are long stretches with the ball missing and
  players walking (< ~1.2 m/s) where no rule fires — likely whistle stoppages (fouls, injuries).
- **Visible over-runs:** some stoppages continue while players are already running at 3–4 m/s:
  the restart is found late when the ball is still missing after play resumes.

### Next
1. **Check by eye** the samples in 2e and 3d in the video.
2. **Refine the dead-ball rules:** add *ball missing + players walking for > ~10 s* (whistles),
   and end a stoppage when players accelerate, not only when the ball is seen moving.
3. Then the **H/A player-movement test** on the 25k labelled frames.